In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the cleaned sleep dataset
df = pd.read_csv('../data/processed/sleep_cleansed.csv')

# 2. Separate Features (X) and Target (y)
X = df.drop(columns=['Target_Sleep_Disorder'])
y = df['Target_Sleep_Disorder']

# 3. Stratified Train/Test Split (80/20)
# Stratify ensures the None, Insomnia, and Sleep Apnea classes are balanced in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Data Split Complete ---")
print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")

--- Data Split Complete ---
Training Features Shape: (299, 12)
Testing Features Shape: (75, 12)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. Define which columns need which preprocessing
categorical_cols = ['Gender', 'Occupation', 'BMI Category']
numerical_cols = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 
                  'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic_BP', 'Diastolic_BP']

# 2. Build the Preprocessor
# StandardScaler ensures large numbers (Daily Steps) don't overpower small numbers (Sleep Duration)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# 3. Build and Train the Pipeline with Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

# 4. Predict and Evaluate on the unseen Test Set
y_pred_lr = lr_pipeline.predict(X_test)

print("--- Logistic Regression Results ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Macro F1: {f1_score(y_test, y_pred_lr, average='macro'):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['None', 'Insomnia', 'Sleep Apnea']))

--- Logistic Regression Results ---
Accuracy: 0.8933
Macro F1: 0.8663

Classification Report:
              precision    recall  f1-score   support

        None       1.00      0.91      0.95        44
    Insomnia       0.92      0.80      0.86        15
 Sleep Apnea       0.68      0.94      0.79        16

    accuracy                           0.89        75
   macro avg       0.87      0.88      0.87        75
weighted avg       0.92      0.89      0.90        75



In [3]:
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# 1. XGBoost strictly requires targets to be numeric (0, 1, 2)
# We will use LabelEncoder to ensure y is properly formatted for all models
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Define our advanced models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='mlogloss')
}

best_model_name = "Logistic Regression"
best_macro_f1 = f1_score(y_test, y_pred_lr, average='macro')
best_pipeline = lr_pipeline # from the previous cell

print("--- Advanced Models Evaluation ---\n")

# 3. Train and compare
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train the model
    pipeline.fit(X_train, y_train_enc)
    y_pred = pipeline.predict(X_test)
    
    acc = accuracy_score(y_test_enc, y_pred)
    macro_f1 = f1_score(y_test_enc, y_pred, average='macro')
    
    print(f"{name}:")
    print(f"Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}\n")
    
    # Check if this model beats our current best
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_model_name = name
        best_pipeline = pipeline

print(f"🏆 Overall Best Model: {best_model_name} (Macro F1: {best_macro_f1:.4f})")

# 4. Save the winning pipeline for Streamlit deployment
os.makedirs("../models", exist_ok=True)
model_path = os.path.join("..", "models", "sleep_disorder_model.pkl")
joblib.dump(best_pipeline, model_path)
print(f"✅ Winning model successfully saved to: {model_path}")

--- Advanced Models Evaluation ---



Random Forest:
Accuracy: 0.9333 | Macro F1: 0.9009



XGBoost:
Accuracy: 0.9067 | Macro F1: 0.8588

🏆 Overall Best Model: Random Forest (Macro F1: 0.9009)
✅ Winning model successfully saved to: ..\models\sleep_disorder_model.pkl


## Honest scoring: why 0.933 is optimistic

The cells above use a **single 80/20 split** and report Random Forest accuracy **0.9333**. That number is not wrong arithmetic, but it is *optimistic by construction*, for two reasons:

1. **A single split** — one lucky partition of 75 test rows. The score depends on which rows happened to land in the test set.
2. **Duplicate feature patterns** — this dataset has 374 rows but only **109 unique feature patterns**. Row-wise splitting lets identical rows appear in *both* train and test, so the model is partly tested on rows it has already seen.

The fix for (2) is **`StratifiedGroupKFold`**, where a *group* = one unique feature pattern, guaranteeing each pattern stays in a single fold.

> This dataset has **synthetic labels** and is an **educational demo** only. The project's evidence base is the CDC NHANES cohort in notebooks 03-05.

In [4]:
import numpy as np, pandas as pd, json
from hashlib import md5
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('../data/processed/sleep_cleansed.csv')
Xk = df.drop(columns=['Target_Sleep_Disorder'])
yk = df['Target_Sleep_Disorder']

# A group = one unique feature pattern. Hash the row's bytes so identical
# rows share a group id.
groups = pd.util.hash_pandas_object(Xk, index=False).values

print(f'rows                   : {len(Xk)}')
print(f'unique feature patterns: {pd.unique(groups).size}')
print(f'duplicate rows         : {Xk.duplicated().sum()}')

rows                   : 374
unique feature patterns: 109
duplicate rows         : 265


In [5]:
# Same estimator as the deployed pickle, so this measures the real model.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

cat = ['Gender', 'Occupation', 'BMI Category']
num = [c for c in Xk.columns if c not in cat]
pre = ColumnTransformer([('num', StandardScaler(), num),
                         ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
demo_pipe = Pipeline([('pre', pre),
                      ('classifier', RandomForestClassifier(random_state=42))])

from sklearn.metrics import f1_score

def score(splitter, **kw):
    accs, f1s = [], []
    for tr, te in splitter.split(Xk, yk, **kw):
        demo_pipe.fit(Xk.iloc[tr], yk.iloc[tr])
        p = demo_pipe.predict(Xk.iloc[te])
        accs.append((yk.iloc[te] == p).mean())
        f1s.append(f1_score(yk.iloc[te], p, average='macro', zero_division=0))
    return np.array(accs), np.array(f1s)

# (1) row-wise CV: duplicates can straddle the split
ra, rf1 = score(RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0))
# (2) group-aware CV: each unique pattern stays in one fold
ga, gf1 = score(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0),
                groups=groups)

def ci(a):
    return f'{a.mean():.4f}  (95% CI {np.percentile(a,2.5):.4f} - {np.percentile(a,97.5):.4f})'

print('Row-wise CV accuracy      :', ci(ra))
print('Group-aware CV accuracy   :', ci(ga))
print(f'Optimistic bias (row - group): {ra.mean()-ga.mean():+.4f}')

Row-wise CV accuracy      : 0.9064  (95% CI 0.8613 - 0.9520)
Group-aware CV accuracy   : 0.9038  (95% CI 0.8573 - 0.9320)
Optimistic bias (row - group): +0.0027


In [6]:
legacy = json.load(open('../results/metrics/metrics.json', encoding='utf-8'))['legacy_demo']
print('Reported in metrics.json:')
print('  n rows                 :', legacy['n_rows'])
print('  unique feature patterns:', legacy['n_unique_feature_patterns'])
print('  row-wise CV accuracy   :', legacy['rowwise_cv_accuracy']['mean'])
print('  group-aware CV accuracy:', legacy['groupaware_cv_accuracy']['mean'])
print('  optimistic bias        :', legacy['optimistic_bias_accuracy'])
assert abs(ga.mean() - legacy['groupaware_cv_accuracy']['mean']) < 0.03, 'group-aware score drifted'
assert pd.unique(groups).size == legacy['n_unique_feature_patterns'], 'pattern count drifted'
print('\nOK - notebook agrees with metrics.json (educational demo, not the evidence base)')

Reported in metrics.json:
  n rows                 : 374
  unique feature patterns: 109
  row-wise CV accuracy   : 0.9064
  group-aware CV accuracy: 0.9038
  optimistic bias        : 0.0027

OK - notebook agrees with metrics.json (educational demo, not the evidence base)
